In [ ]:
from pathlib import Path
import os
import sys
import ast
import importlib

REPO = Path("/workspace/ProtoSAM-repro")
PASTA = REPO / "exp/eft_validacao_polipos_v1"

assert Path(sys.executable).resolve() == Path(
    "/workspace/miniforge3/envs/protosam/bin/python"
).resolve(), sys.executable

os.chdir(REPO)

codigo = (PASTA / "validar_polipos.py").read_text()

def trocar(antes, depois):
    global codigo
    assert codigo.count(antes) == 1, (
        f"Avaliador diferente do esperado: {antes}"
    )
    codigo = codigo.replace(antes, depois, 1)

trocar(
    "canais = VARIANTES[variante]['dims'] * 8",
    "canais = VARIANTES[variante]['dims'] * 4",
)

trocar(
    """            fts = self.encoder(imgs_concat)[-1]
            assert tuple(fts.shape[1:]) == (canais,21,21)""",
    """            fts = self.encoder.patch_embed(imgs_concat)
            for i in range(3):
                fts = self.encoder.layers[i].blocks(fts)
                if i < 2:
                    fts = self.encoder.layers[i].downsample(fts)

            assert self.encoder.channel_first
            assert tuple(fts.shape[1:]) == (canais,42,42)

            fts = F.layer_norm(
                fts.permute(0,2,3,1), (canais,),
                eps=self.encoder.outnorm3.eps,
            ).permute(0,3,1,2).contiguous()""",
)

trocar(
    "def executar(piloto, retomar=None):",
    "def executar(piloto, passo, baseline, retomar=None):",
)

trocar(
    "    assert resumo_piloto['atualizacoes'] == 200",
    """    assert resumo_piloto['atualizacoes'] == 5000 and variante == 'tiny'
    assert cfg_piloto['estagio'] == 2
    assert cfg_piloto['normalizacao'] == (
        'LayerNorm fixa escala 1 vies 0; antes de interpolar'
    )
    assert passo in (1000,1500,2000,3000,4000,5000)""",
)

trocar(
    "ckpt_eft = piloto / 'checkpoint_latest.pt'",
    "ckpt_eft = piloto / f'checkpoint_{passo:04d}.pt'",
)

trocar(
    "salvo['passo'] == 200 and",
    "salvo['passo'] == passo and",
)

trocar(
    "'resolucao_encoder':672, 'canvas':1024, 'estagio':3,",
    "'resolucao_encoder':672, 'canvas':1024, 'estagio':2,\n"
    "        'passo':passo, 'normalizacao_experimento':'ln_unitaria',",
)

trocar(
    "'norm_antes':'classifier.norm pré-treinada transferida para outnorm3',",
    "'norm_antes':'ln_unitaria',",
)

trocar(
    "'norm_depois':'outnorm3 do checkpoint EFT',",
    "'norm_depois':'ln_unitaria',",
)

trocar(
    "    pasta = Path(retomar).resolve() if retomar else "
    "REPO/'reproducao'/f'eft_{variante}_validacao_polipos_{stamp()}'",
    """
    baseline = Path(baseline).resolve()
    ref = ler(baseline / 'config.json')

    for chave in (
        'variante','episodios','suporte','resolucao_encoder','canvas',
        'estagio','normalizacao_experimento','norm_antes','norm_depois',
        'sam','selecao_sam','alp_inferencia','grade_features',
        'kernel_configurado','checkpoint_original_sha256',
        'checkpoint_sam_sha256','gpu_driver','torch','cuda','versoes',
        'suporte_tensor_sha256','cudnn','tf32_matmul','tf32_cudnn',
    ):
        assert ref[chave] == config[chave], (
            f'Baseline incompatível: {chave}'
        )

    for fonte, digest in config['fontes_sha256'].items():
        if fonte in ref['fontes_sha256']:
            assert ref['fontes_sha256'][fonte] == digest, fonte

    config['baseline_reutilizada'] = str(baseline)
    config['baseline_config_sha256'] = sha(baseline / 'config.json')

    pasta = (
        Path(retomar).resolve()
        if retomar
        else REPO/'reproducao'/f'eft_penultima_passo_{passo}_validacao_{stamp()}'
    )
""",
)

trocar(
    "    modelo = construir_672(variante)",
    """
    if not (pasta / 'antes').exists():
        temporario = pasta / f'antes_tmp_{stamp()}'
        shutil.copytree(baseline / 'antes', temporario)
        temporario.rename(pasta / 'antes')

    modelo = construir_672(variante)
""",
)

trocar(
    "resumo={'variante':variante,'n':145,",
    "resumo={'variante':variante,'passo':passo,'estagio':2,'n':145,",
)

ast.parse(codigo)
destino = PASTA / "validar_penultima_marcos.py"

if destino.exists():
    assert destino.read_text() == codigo, (
        f"Já existe outra versão: {destino}"
    )
else:
    destino.write_text(codigo)

for caminho in (REPO, PASTA):
    if str(caminho) in sys.path:
        sys.path.remove(str(caminho))
    sys.path.insert(0, str(caminho))

importlib.invalidate_caches()
sys.modules.pop("validar_penultima_marcos", None)

from validar_penultima_marcos import executar, ler, salvar

print("Avaliador preparado. Nenhuma avaliação iniciada.")

In [ ]:
import json

TREINO = REPO / (
    "reproducao/eft_tiny_penultima_ln_5000_20261012T210006906429Z"
)

CONTROLE_CAMADAS = REPO / (
    "reproducao/avaliacao_camadas_norm_tiny_2000_v1.json"
)

# Reutiliza apenas a condição "antes": penúltima, LN fixa, pré-treinado.
BASELINE = Path(
    ler(CONTROLE_CAMADAS)["avaliacoes"]["camada2_ln_unitaria"]
)

REFERENCIA_ULTIMA = REPO / (
    "reproducao/eft_tiny_referencia_driver_atual_20261012T215801165279Z"
)

CONTROLE = REPO / (
    "reproducao/avaliacao_penultima_20261012T210006906429Z.json"
)

PASSOS = (1000, 1500, 2000, 3000, 4000, 5000)

for pasta in (TREINO, BASELINE, REFERENCIA_ULTIMA):
    assert (pasta / "resumo.json").is_file(), pasta

for passo in PASSOS:
    checkpoint = TREINO / f"checkpoint_{passo:04d}.pt"
    assert checkpoint.is_file(), checkpoint

estado = (
    ler(CONTROLE)
    if CONTROLE.exists()
    else {
        "treino": str(TREINO),
        "baseline": str(BASELINE),
        "avaliacoes": {},
    }
)

assert estado["treino"] == str(TREINO)
assert estado["baseline"] == str(BASELINE)

for passo in PASSOS:
    chave = str(passo)
    retomar = estado["avaliacoes"].get(chave)

    if retomar is None:
        candidatas = []

        for arquivo in (REPO / "reproducao").glob(
            f"eft_penultima_passo_{passo}_validacao_*/config.json"
        ):
            cfg = ler(arquivo)
            if cfg.get("piloto") == str(TREINO):
                candidatas.append(arquivo.parent)

        assert len(candidatas) <= 1, (
            f"Mais de uma avaliação para {passo}: {candidatas}"
        )
        if candidatas:
            retomar = str(candidatas[0])

    print(f"\n===== PENÚLTIMA: {passo} PASSOS =====", flush=True)

    resultado = executar(
        piloto=TREINO,
        passo=passo,
        baseline=BASELINE,
        retomar=retomar,
    )

    estado["avaliacoes"][chave] = str(resultado)
    salvar(CONTROLE, estado)

inicial = ler(BASELINE / "resumo.json")
comparacao = [{
    "passo": 0,
    "Dice_ALP": inicial["ALP"]["Dice"]["antes"],
    "Dice_SAM": inicial["SAM"]["Dice"]["antes"],
}]

for passo in PASSOS:
    resumo = ler(
        Path(estado["avaliacoes"][str(passo)]) / "resumo.json"
    )
    comparacao.append({
        "passo": passo,
        "Dice_ALP": resumo["ALP"]["Dice"]["depois"],
        "Dice_SAM": resumo["SAM"]["Dice"]["depois"],
    })

referencia = ler(REFERENCIA_ULTIMA / "resumo.json")

estado["comparacao_penultima"] = comparacao
estado["referencia_ultima_EFT_2000"] = {
    "Dice_ALP": referencia["ALP"]["Dice"]["depois"],
    "Dice_SAM": referencia["SAM"]["Dice"]["depois"],
    "pasta": str(REFERENCIA_ULTIMA),
}
salvar(CONTROLE, estado)

print("\nCOMPARAÇÃO FINAL — TREINO PELA PENÚLTIMA")
print(json.dumps(comparacao, indent=2, ensure_ascii=False))

print("\nREFERÊNCIA — ÚLTIMA, EFT 2.000")
print(json.dumps(
    estado["referencia_ultima_EFT_2000"],
    indent=2,
    ensure_ascii=False,
))

print("\nRegistro:", CONTROLE)

In [ ]:
from pathlib import Path
import json
import sys
import torch

REPO = Path("/workspace/ProtoSAM-repro")
TREINO = REPO / (
    "reproducao/eft_tiny_penultima_ln_5000_20261012T210006906429Z"
)

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA do PyTorch:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version())
print("GPU:", torch.cuda.get_device_name())

for passo in (1000, 1500, 2000, 3000, 4000, 5000):
    encontradas = []

    for arquivo in (REPO / "reproducao").glob(
        f"eft_penultima_passo_{passo}_validacao_*/config.json"
    ):
        cfg = json.loads(arquivo.read_text())
        if cfg.get("piloto") != str(TREINO):
            continue

        pasta = arquivo.parent
        quantidade = len(list((pasta / "depois/registros").glob("*.json")))

        encontradas.append({
            "pasta": str(pasta),
            "imagens_registradas": quantidade,
            "resumo_concluido": (pasta / "resumo.json").is_file(),
            "gpu_driver_anterior": cfg.get("gpu_driver"),
            "torch_anterior": cfg.get("torch"),
            "cuda_anterior": cfg.get("cuda"),
        })

    print(f"\nPASSO {passo}")
    print(json.dumps(encontradas, indent=2, ensure_ascii=False))

In [ ]:
from pathlib import Path
import os
import sys
import ast
import importlib

REPO = Path("/workspace/ProtoSAM-repro")
PASTA = REPO / "exp/eft_validacao_polipos_v1"
os.chdir(REPO)

assert Path(sys.executable).resolve() == Path(
    "/workspace/miniforge3/envs/protosam/bin/python"
).resolve(), sys.executable

fonte = PASTA / "validar_penultima_marcos.py"
codigo = fonte.read_text()

# Permite que a baseline reutilizada tenha sido calculada em outra GPU.
# Todas as demais verificações de dados, pesos e ambiente permanecem.
trecho = "'gpu_driver',"
assert codigo.count(trecho) == 1
codigo = codigo.replace(trecho, "", 1)

antes = "    config['baseline_reutilizada'] = str(baseline)"
depois = """    config['baseline_gpu_driver'] = ref['gpu_driver']
    config['baseline_em_outro_ambiente'] = (
        ref['gpu_driver'] != config['gpu_driver']
    )
    print('GPU/driver da baseline:', ref['gpu_driver'], flush=True)
    print('GPU/driver desta avaliação:', config['gpu_driver'], flush=True)
    config['baseline_reutilizada'] = str(baseline)"""

assert codigo.count(antes) == 1
codigo = codigo.replace(antes, depois, 1)

antes = "f'eft_penultima_passo_{passo}_validacao_{stamp()}'"
depois = "f'eft_penultima_passo_{passo}_gpu4090_{stamp()}'"

assert codigo.count(antes) == 1
codigo = codigo.replace(antes, depois, 1)

ast.parse(codigo)

destino = PASTA / "validar_penultima_4090.py"
if destino.exists():
    assert destino.read_text() == codigo, (
        f"Já existe outra versão: {destino}"
    )
else:
    destino.write_text(codigo)

for caminho in (REPO, PASTA):
    if str(caminho) in sys.path:
        sys.path.remove(str(caminho))
    sys.path.insert(0, str(caminho))

importlib.invalidate_caches()
sys.modules.pop("validar_penultima_4090", None)

from validar_penultima_4090 import executar, ler, salvar

print("Preparado. Os arquivos e resultados anteriores foram preservados.")

In [ ]:
import json
import torch

assert "4090" in torch.cuda.get_device_name(), (
    "Este conjunto de avaliações foi preparado para a RTX 4090."
)

TREINO = REPO / (
    "reproducao/eft_tiny_penultima_ln_5000_20261012T210006906429Z"
)

BASELINE = Path(
    ler(REPO / "reproducao/avaliacao_camadas_norm_tiny_2000_v1.json")
    ["avaliacoes"]["camada2_ln_unitaria"]
)

REFERENCIA_ULTIMA = REPO / (
    "reproducao/eft_tiny_referencia_driver_atual_20261012T215801165279Z"
)

CONTROLE = REPO / (
    "reproducao/avaliacao_penultima_20261012T210006906429Z_4090.json"
)

CONCLUIDOS = {
    "1000": str(REPO / (
        "reproducao/eft_penultima_passo_1000_validacao_20261012T235022253086Z"
    )),
    "1500": str(REPO / (
        "reproducao/eft_penultima_passo_1500_validacao_20261013T001309011868Z"
    )),
    "2000": str(REPO / (
        "reproducao/eft_penultima_passo_2000_validacao_20261013T003337707573Z"
    )),
}

for passo, caminho in CONCLUIDOS.items():
    pasta = Path(caminho)
    assert (pasta / "resumo.json").is_file(), pasta
    cfg = ler(pasta / "config.json")
    assert cfg["piloto"] == str(TREINO)
    assert cfg["passo"] == int(passo)

estado = (
    ler(CONTROLE)
    if CONTROLE.exists()
    else {
        "treino": str(TREINO),
        "baseline": str(BASELINE),
        "avaliacoes": dict(CONCLUIDOS),
    }
)

assert estado["treino"] == str(TREINO)
assert estado["baseline"] == str(BASELINE)

for passo in (3000, 4000, 5000):
    assert (TREINO / f"checkpoint_{passo:04d}.pt").is_file()

    chave = str(passo)
    retomar = estado["avaliacoes"].get(chave)

    if retomar is None:
        candidatas = []
        for arquivo in (REPO / "reproducao").glob(
            f"eft_penultima_passo_{passo}_gpu4090_*/config.json"
        ):
            cfg = ler(arquivo)
            if cfg.get("piloto") == str(TREINO):
                candidatas.append(arquivo.parent)

        assert len(candidatas) <= 1, (
            f"Mais de uma avaliação para {passo}: {candidatas}"
        )
        if candidatas:
            retomar = str(candidatas[0])

    print(f"\n===== PENÚLTIMA: {passo} PASSOS — RTX 4090 =====", flush=True)

    resultado = executar(
        piloto=TREINO,
        passo=passo,
        baseline=BASELINE,
        retomar=retomar,
    )

    estado["avaliacoes"][chave] = str(resultado)
    salvar(CONTROLE, estado)

inicial = ler(BASELINE / "resumo.json")
comparacao = [{
    "passo": 0,
    "Dice_ALP": inicial["ALP"]["Dice"]["antes"],
    "Dice_SAM": inicial["SAM"]["Dice"]["antes"],
    "gpu_driver": ler(BASELINE / "config.json")["gpu_driver"],
}]

for passo in (1000, 1500, 2000, 3000, 4000, 5000):
    pasta = Path(estado["avaliacoes"][str(passo)])
    resumo = ler(pasta / "resumo.json")
    cfg = ler(pasta / "config.json")

    comparacao.append({
        "passo": passo,
        "Dice_ALP": resumo["ALP"]["Dice"]["depois"],
        "Dice_SAM": resumo["SAM"]["Dice"]["depois"],
        "gpu_driver": cfg["gpu_driver"],
    })

referencia = ler(REFERENCIA_ULTIMA / "resumo.json")
estado["comparacao_penultima"] = comparacao
estado["referencia_ultima_EFT_2000"] = {
    "Dice_ALP": referencia["ALP"]["Dice"]["depois"],
    "Dice_SAM": referencia["SAM"]["Dice"]["depois"],
    "gpu_driver": ler(REFERENCIA_ULTIMA / "config.json")["gpu_driver"],
}
salvar(CONTROLE, estado)

print("\nCOMPARAÇÃO FINAL — TREINO PELA PENÚLTIMA")
print(json.dumps(comparacao, indent=2, ensure_ascii=False))

print("\nREFERÊNCIA — ÚLTIMA, EFT 2.000")
print(json.dumps(
    estado["referencia_ultima_EFT_2000"],
    indent=2,
    ensure_ascii=False,
))
print("\nRegistro:", CONTROLE)